In [1]:
import os
# Setează numărul de nuclee dorit (ex: 16)
n_core = "16"
os.environ["OMP_NUM_THREADS"] = n_core
os.environ["MKL_NUM_THREADS"] = n_core
os.environ["OPENBLAS_NUM_THREADS"] = n_core
os.environ["VECLIB_MAXIMUM_THREADS"] = n_core
os.environ["NUMEXPR_NUM_THREADS"] = n_core

import numpy as np
import warnings
# suppress warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import multilabel_confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

# Pentru metrice

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score

import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# data loading
data_clean = pd.read_csv("data_clean.csv")
data_clean.shape

(9719, 17)

### Proportionality
$$p_i = \frac{n_i}{N}$$
Unde:
* **$p_i$**: Class probability or proportion $i$.
* **$n_i$**: Number of elements (observations) in the class $i$.
* **$N$**: Total number of elements in the dataset $S$ (unde $N = \sum n_i$).

### Entropia
$$H(S) = -\sum_{i=1}^{c} p_i \log_2(p_i)$$
$$H(S) = - ( p_1 \log_2 p_1 + p_2 \log_2 p_2 + ... + p_i \log_2 p_i )$$
Unde:
* $c$: Total number of classes.
* $p_1, p_2, p_i$:  Represents the probability (relative frequency) of each class.
* The basic condition is that **$\sum p_i = 1$**.

### Information Gain (IG)
$$IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v)$$

Unde:
* **$H(S)$**: Entropy of the dataset before split (Parent Node).
* **$A$**: The attribute (characteristic) based on which the split is made.
* **$v$**: A specific attribute value $A$.
* **$|S_v|$**: Number of elements in the sub-setul result for the value $v$.
* **$|S|$**: Total number of elements in the parent node.
* **$H(S_v)$**: Entropy of the resulting child node.

### Gini Impurity
$$ Gini(S) = 1 - \sum_{i=1}^{c} p_i^2 $$
$$Gini(S) = 1 - (p_1^2 + p_2^2 + ... + p_i^2)$$
* $Gini(S) = 0$ (All elements belong to a single class).
* $Gini(S) = 1−(1/c)$ (Maximum uncertainty)

In [3]:
class DecisionMetrics:

    @staticmethod
    # Probabilitatea fiecarei clase unice dintr-un set de date.
    # Exemplu: Pentru [0, 0, 1, 1, 1], proportiile sunt [0.4, 0.6].
    def get_proportions(y):

        if len(y) == 0: 
            return np.array([])# Returneaza un array gol daca nu sunt date

        # Cauta de cate ori apare fiecare clasă
        _, counts = np.unique(y, return_counts=True)

        # Calculeaza proportionalitatea (frecventa relativa)
        return counts / len(y)

    @staticmethod
    # Calculeaza dezordinea datelor (Entropy) sau a modului in care energia se disperseaza intr-un sistem
    # 0 = Puritate maxima
    # entropia maxima este log2(numărul de clase) ... toate clasele sunt perfect egale ca proportie (confuzie totala)
    def entropy(y):
        p = DecisionMetrics.get_proportions(y)
        return -np.sum(p * np.log2(p + 1e-10))


    @staticmethod
    # Impuritatea Gini masoara probabilitatea de a eticheta gresit unul sau mai multe elemente dintr-un grup.
    # 0 = Puritate maxima
    # 1−(1/numărul de clase) ... toate clasele sunt perfect egale ca proportie (confuzie totala)
    def gini_impurity(y):
        p = DecisionMetrics.get_proportions(y)
        if len(p) == 0:
            return 0.0 # in cazul in care y este gol
        return 1 - np.sum(p**2)

    @staticmethod
    # Information Gain masoara cata ordine am reusit să facem dupa ce am aplicat o regulă de separare.
    def information_gain(y_parent, y_left, y_right, mode='entropy'): #mode='entropy'

        if mode == 'entropy':
            metric_fn = DecisionMetrics.entropy
        else:
            metric_fn = DecisionMetrics.gini_impurity

        # Impuritatea Gini initiala
        parent_loss = metric_fn(y_parent)

        # Proportaonalitatea copiilor (cat merg în stânga/dreapta)
        n_total = len(y_parent)
        w_y_left = len(y_left) / n_total       # ponderea din stanga
        w_y_right = len(y_right) / n_total     # ponderea din dreapta
        
        # Impuritatea Gini dupa taietura
        gini_y_left = metric_fn(y_left)
        gini_y_right = metric_fn(y_right)

        
        # Media ponderata a impuritatii copiilor ()
        child_loss = (w_y_left * gini_y_left) + (w_y_right * gini_y_right)

        # Information Gain rezulta din diferenta
        return parent_loss - child_loss


In [4]:
class DecisionStump:
    
    def __init__(self, min_samples_leaf, criterion='entropy'):
        self.min_samples_leaf = min_samples_leaf # valoarea minima se exemple pentru care va fi luata in calcul o ramura
        self.criterion = criterion
        self.best_feature_idx = None  # cea mai buna caracteristica (coloana)
        self.best_threshold = None    # cea mai buna valoare de taiere
        self.best_gain = -1 # valoarea celui mai bun castig
        self.class_left = None  # valoarea returnata daca <= threshold (True)
        self.class_right = None  # valoarea returnata daca > threshold ()

    # Invata care este cea mai buna intrebare
    def fit(self, X, y):

        # Extrage numarul de exemple (coloana) si numarul de caracteristici (lini)
        n_samples, n_features = X.shape

        # Itereaza pachet de caracteristici
        for f_idx in range(n_features):
            # Extrage valorile unice si le defineste ca praguri de testare 
            thresholds = np.unique(X[:, f_idx])

            # Itereaza fiecare prag si inparte datele in doua: Stanga (sub prag) si Dreapta (peste prag)
            for threshold in thresholds:
                # Creaza o masca de forma True - False, pentru datele din stanga (datele de sub threshold) 
                left_mask = X[:, f_idx] <= threshold 
                # Separa datele din stanga si dreapta prin aplicarea mastilor
                y_left, y_right = y[left_mask], y[~left_mask]
                # In cazul in care dimensiunea de date, din stanga sau dreapta este nul,
                # sare peste acea valoare a "threshold", 
                # deoarece aceasta valoare de "threshold" nuta toate rezultatele in stanga sau in dreapta  
                if y_left.size <= self.min_samples_leaf or y_right.size <= self.min_samples_leaf: 
                    continue

                # Apeleaza metoda de calcul pentru a obtine o valore "gain" a eficientei generata de valoarea 
                # de "threshold" din iteratia actuala
                gain = DecisionMetrics.information_gain(y, y_left, y_right, mode=self.criterion)

                # Verifica daca eficienta obtinuta cu aceasta valoare de "threshold" este cea mai buna de pana acum
                # In cazul in care eficienta este mai buna, salveaza parametrii
                if gain > self.best_gain:
                    self.best_gain = gain # actualizeaza valoarea celui mai bun castig
                    self.best_feature_idx = f_idx # identifica cea mai buna caracteristica (coloana)
                    self.best_threshold = threshold # memoreaza valoarea celui mai bun prag
                    
                    # Salveaza deciziile luate pe fiecare ramura 
                    # --clasa majoritara pentru frunza stanga si pentru frunza dreapta --
                    self_class_left = np.bincount(y_left) # identifica cate valori identice (cate clase sunt)
                    self.class_left = self_class_left.argmax() # identifica numarul maxim (clasa predominanta)
                    self.class_right = np.bincount(y_right).argmax()
                    

    # Pune datele pe ranura corecta (stânga sau dreapta)
    def predict(self, X):
        # Extragem doar coloana cu caracteristicile cele mai concludente pentru split-ul
        best_column = X[:, self.best_feature_idx] 

        predictions =[]
        # Parcurgerea fiecărei valori din coloana optimă
        for value in best_column:
            # Verificarea condiției de split
            if value <= self.best_threshold:    
                predictions.append(self.class_left) # True pentru stanga (insereaza clasa majora din stanga)
            else:
                predictions.append(self.class_right) # False pentru dreapta (insereaza clasa majora din dreapta)
        
        # Converteste lista in array pentru consistenta
        results = np.array(predictions)
        return results


    def __repr__(self):
        if self.best_feature_idx is None:
            return "DecisionStump(Neantrenat)"
        return (f"DecisionStumpMulticlass(feature_idx={self.best_feature_idx}, "
                f"best_gain={self.best_gain}, "
                f"best_feature_idx={self.best_feature_idx}, "
                f"threshold={self.best_threshold:.4f}, "
                f"class_left={self.class_left}, "
                f"class_right={self.class_right})")

In [5]:
exemple = 500
caracteristici = 90

np.random.seed(48)
X = np.random.randn(exemple, caracteristici)
y = np.random.randint(0, 7, size=exemple)

stump = DecisionStump(30,criterion='gini_impurity')
stump.fit(X, y)
print(stump)

# Predicție pe un set nou
X_new = np.random.randn(30, caracteristici)
print(f"Predicții finale: {stump.predict(X_new)}")

DecisionStumpMulticlass(feature_idx=66, best_gain=0.007715484988575216, best_feature_idx=66, threshold=1.4081, class_left=2, class_right=6)
Predicții finale: [2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 6 2 2 2 2 2 2 6 2 2 2 2 2 2 2]


In [6]:
# Celula de memorie a arborelui decizional 
# Un nod poate fi o intrebare (split) sau un rezultat final (frunza)
# Stocheza structura ierarhică pe care algoritmul o învață în timpul antrenării (fit)
# Este folosita doar in timpul antrenamentului pentru ca algoritmul sa descopere care sunt cele mai bune praguri
class Node:

    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature      # caracteristica, adica, coloana care se verifica
        self.threshold = threshold  # valoarea de prag 
        self.left = left            # ramura pentru "True" (stanga)
        self.right = right          # ramura pentru "False" (dreapta)
        self.value = value          # Daca e frunza, aici stocam clasa prezisa

    def is_leaf_node(self):
        # Un nod este 'frunza' daca are o valoare finala, deci nu mai are copii
        # Verifica daca in "value" avem o predictie salvata 
        if self.value is not None:
            return True   # Da, este un nod frunză
        else:
            return False  # Nu, este un nod de decizie (value este None)
        
         
class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2, mode ='entropy'):
        self.max_depth = max_depth                 # cat de adanc poate creste arborele
        self.min_samples_split = min_samples_split # minim cate date trebuie sa mai fie pentru a face un split
        self.root = None                           # punctul de start al arborelui
        self.mode = mode                           # Modul in care se calculeaza 'entropy' sau 'gini_impurity'

    # Conventie: toate modelele trebuie sa aiba o metoda numita .fit(X, y).
    def fit(self, X, y):
        # self.root conține întreaga ierarhie în memorie a arborelui,
        # de la prima întrebare până la ultimul răspuns
        # sub forma unui singur obiect
        self.root = self._grow_tree(X, y)

    # Functie recursiva care "fragmenteaza" repetat setul de date în grupuri 
    # din ce în ce mai mici si mai pure, 
    # Construieste ierarhia de decizii a arborelui pana cand ajunge la un raspuns final (frunza)   
    def _grow_tree(self, X, y, depth=0):
        n_samples, n_features = X.shape # extrage numarul de exemple si de caracteristici (lini si coloane)
        n_labels = len(np.unique(y))    # extrage numarul valorilor unice din etichete (determina numarul de clase)

        # Conditie de orire daca :
        # - daca ajuns la adâncimea maxima, 
        # - daca toate datele sunt din aceeași clasa,
        # - daca sunt prea putine date pentru a mai face o impartire logica.
        if (depth >= self.max_depth 
            or n_labels == 1 
            or n_samples < self.min_samples_split):
            
            leaf_value = self._most_common_label(y) # returneaza clasa majora
            return Node(value=leaf_value)

        # Cautam prin toate caracteristicile (coloanele) iterand toate valorile posibile pentru a gasi cea mai buna sectiune
        # returneaza indexul celei mai bune caracteristici (coloanei) si valoarea celui mai bun prag
        best_feat, best_thresh = self._best_split(X, y, n_features) 

        # Creaza ramuri
        # In functie de "best_thresh" inparte datele (best_feat) in True si False (stanga si dreapta)
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh) # returneaza indexul datelor inpartite in ramura stanga si dreapta
        
        # Recursivitate, metoda "_grow_tree" se lanseaza in interiorul ei pentru a crea sub-arborii stang și drept
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1) # "depth + 1" incrementeaza noul nivel
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)

        # Memoreaza in clasa "Node" noua arhitectura
        return Node(feature=best_feat, threshold=best_thresh, left=left, right=right)

    # Gaseste cea mai buna sectiune
    # Itereaza prin toate trasaturile pentru a gasi 
    def _best_split(self, X, y, n_features):
        best_gain = -1
        split_idx, split_thresh = None, None

        # Itereaza toate caracteristicile
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column) # extrage toate valorile unice din coloana ca posibile praguri

            # Itereaza toate pragurile unice extrase din "feat_idx"
            for threshold in thresholds:

                # Calculeaza cat de  "curate" sunt datele acestui split
                y_left_i, y_right_i=self._split(X_column, threshold)
                gain = DecisionMetrics.information_gain(y, y[y_left_i], y[y_right_i], mode=self.mode)

                # Verifica daca actualul "gain" este cel mai bun
                if gain > best_gain:
                    # in cazul in care acuala valore de prag este mai buna, se fac salvarile pentru :
                    best_gain = gain           # "gain"
                    split_idx = feat_idx       # indexul caracteristici pe care sa realizat iteratia
                    split_thresh = threshold   # valoarea pragului

        return split_idx, split_thresh

    # Imparte indexurile datelor In funcție de prag
    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten() # np.argwhere - gaseste coordonatele elementelor 
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    # Determina clasa majoritara (folosita in nodurile frunza)
    def _most_common_label(self, y):
        return np.bincount(y).argmax()

    # Primeste o lista de caracteristici si si face predictii
    # aici se lanseaza predictia
    def predict(self, X):
        rezultate = []
        # Itereaza randurile din setul de date
        for x in X:
            # Parcurgi elementul prin arbore pentru a returna predictia
            pred = self._traverse_tree(x, self.root)
            rezultate.append(pred)
        return np.array(rezultate)

    # Parcurge arborele de la radacina la frunza pentru un singur rand de date
    def _traverse_tree(self, x, node):
        
        if node.is_leaf_node():
            return node.value # raspuns final si oprim cautarea
        
        # Mergem la stânga sau la dreapta în funcție de valoarea trăsăturii
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [7]:
exemple = 500
caracteristici = 4

np.random.seed(48)
X = np.random.randn(exemple, caracteristici)
y = np.random.randint(0, 7, size=exemple)

decision_tree = DecisionTree( max_depth=5, min_samples_split=3, mode='gini_impurity')
decision_tree.fit(X, y)


# Predicție pe un set nou
X_new = np.random.randn(30, caracteristici)
print(f"Predicții finale: {decision_tree.predict(X_new)}")

Predicții finale: [6 5 1 6 5 4 5 4 4 5 1 0 2 6 6 3 1 1 5 4 1 6 5 6 3 6 6 3 6 1]


In [8]:
# Structura arborelui decizional
def print_tree(node, depth=0): # depth=0 este o variabila de spatiere

    # Daca sa ajuns la o frunza, returneaza valoarea
    if node.is_leaf_node():
        indent = "  " * depth
        print(f"{indent}└── [Frunza: Clasa {node.value}]")
        return

    # Daca este nod de decizie, se afiseaza regula (feature si threshold)
    indent = "  " * depth
    print(f"{indent}├── [Nivel {depth}] Daca Feature {node.feature} <= {node.threshold}:")
    
    # Apelam recursiv pentru copii, incrementand depth + 1 
    print_tree(node.left, depth + 1)    # pentru stanga
    print_tree(node.right, depth + 1)   # pentru dreapta

print_tree(decision_tree.root)


├── [Nivel 0] Daca Feature 1 <= 1.5405061155759983:
  ├── [Nivel 1] Daca Feature 3 <= -0.4864733641112821:
    ├── [Nivel 2] Daca Feature 3 <= -1.0495115153701928:
      ├── [Nivel 3] Daca Feature 2 <= -0.15438678682604426:
        ├── [Nivel 4] Daca Feature 2 <= -0.497159334300221:
          └── [Frunza: Clasa 0]
          └── [Frunza: Clasa 3]
        ├── [Nivel 4] Daca Feature 1 <= -1.8251252785507444:
          └── [Frunza: Clasa 5]
          └── [Frunza: Clasa 2]
      ├── [Nivel 3] Daca Feature 2 <= -1.1787059996048368:
        └── [Frunza: Clasa 4]
        ├── [Nivel 4] Daca Feature 2 <= 0.2628702733489987:
          └── [Frunza: Clasa 6]
          └── [Frunza: Clasa 4]
    ├── [Nivel 2] Daca Feature 0 <= -0.15740415138658664:
      ├── [Nivel 3] Daca Feature 3 <= 0.3801406544096387:
        ├── [Nivel 4] Daca Feature 3 <= -0.4126503977657889:
          └── [Frunza: Clasa 1]
          └── [Frunza: Clasa 6]
        ├── [Nivel 4] Daca Feature 2 <= -1.2063115458565281:
          └─